In [1]:
%pip install -q langgraph==0.2.57 langchain-ibm==0.3.10

  DEPRECATION: Building 'ibm-cos-sdk' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'ibm-cos-sdk'. Discussion can be found at https://github.com/pypa/pip/issues/6334
  DEPRECATION: Building 'ibm-cos-sdk-core' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'ibm-cos-sdk-core'. Discussion can be found at https://github.com/pypa/pip/issues/6334
  DEPRECATION: Building 'ibm-cos-sdk-s3transfer' using the

In [1]:
from langgraph.graph import StateGraph

In [2]:
from typing import TypedDict, Optional

class AuthState(TypedDict):
    username: Optional[str] 
    password: Optional[str]
    is_authenticated: Optional[bool]
    output: Optional[str]

In [3]:
auth_state_1: AuthState = {
    "username": "alice123",
    "password": "123",
    "is_authenticated": True,
    "output": "Login successful."
}
print(f"auth_state_1: {auth_state_1}")

auth_state_1: {'username': 'alice123', 'password': '123', 'is_authenticated': True, 'output': 'Login successful.'}

In [4]:
auth_state_2: AuthState = {
    "username":"",
    "password": "wrongpassword",
    "is_authenticated": False,
    "output": "Authentication failed. Please try again."
}
print(f"auth_state_2: {auth_state_2}")

auth_state_2: {'username': '', 'password': 'wrongpassword', 'is_authenticated': False, 'output': 'Authentication failed. Please try again.'}

In [5]:
def input_node(state):
    print(state)
    if state.get('username', "") =="":
        username = input("What is your username?")

    password = input("Enter your password: ")
    
    if state.get('username', "") =="":
        return {"username":username, "password":password}
    else:
        return {"password": password}

In [6]:
input_node(auth_state_1)

{'username': 'alice123', 'password': '123', 'is_authenticated': True, 'output': 'Login successful.'}

Enter your password:  helloworld


{'password': 'helloworld'}

In [7]:
input_node(auth_state_2)

{'username': '', 'password': 'wrongpassword', 'is_authenticated': False, 'output': 'Authentication failed. Please try again.'}

What is your username? 
Enter your password:  


{'username': '', 'password': ''}

In [8]:
def validate_credentials_node(state):
    # Extract username and password from the state
    username = state.get("username", "")
    password = state.get("password", "")

    print("Username :", username, "Password :", password)
    # Simulated credential validation
    if username == "test_user" and password == "secure_password":
        is_authenticated = True
    else:
        is_authenticated = False

    # Return the updated state with authentication result
    return {"is_authenticated": is_authenticated}

In [9]:
validate_credentials_node(auth_state_1)

Username : alice123 Password : 123

{'is_authenticated': False}

In [10]:
auth_state_3: AuthState = {
    "username":"test_user",
    "password":  "secure_password",
    "is_authenticated": False,
    "output": "Authentication failed. Please try again."
}
print(f"auth_state_3: {auth_state_3}")

auth_state_3: {'username': 'test_user', 'password': 'secure_password', 'is_authenticated': False, 'output': 'Authentication failed. Please try again.'}

In [11]:
validate_credentials_node(auth_state_3)

Username : test_user Password : secure_password

{'is_authenticated': True}

In [12]:
# Define the success node
def success_node(state):
    return {"output": "Authentication successful! Welcome."}

In [13]:
success_node(auth_state_3)

{'output': 'Authentication successful! Welcome.'}

In [14]:
# Define the failure node
def failure_node(state):
    return {"output": "Not Successfull, please try again!"}

In [15]:
def router(state):
    if state['is_authenticated']:
        return "success_node"
    else:
        return "failure_node"

In [16]:
from langgraph.graph import StateGraph
from langgraph.graph import END

# Create an instance of StateGraph with the GraphState structure
workflow = StateGraph(AuthState)
workflow

In [17]:
workflow.add_node("InputNode", input_node)

In [18]:
workflow.add_node("ValidateCredential", validate_credentials_node)

In [19]:
workflow.add_node("Success", success_node)

In [20]:
workflow.add_node("Failure", failure_node)

In [21]:
workflow.add_edge("InputNode", "ValidateCredential")

In [22]:
workflow.add_edge("Success", END)

In [23]:
workflow.add_edge("Failure", "InputNode")

In [24]:
workflow.add_conditional_edges("ValidateCredential", router, {"success_node": "Success", "failure_node": "Failure"})

In [25]:
workflow.set_entry_point("InputNode")

In [26]:
app = workflow.compile()

In [27]:
inputs = {"username": "test_user"}
result = app.invoke(inputs)
print(result)

{'username': 'test_user'}

Enter your password:  secure_password


Username : test_user Password : secure_password
{'username': 'test_user', 'password': 'secure_password', 'is_authenticated': True, 'output': 'Authentication successful! Welcome.'}

In [28]:
result['output']

'Authentication successful! Welcome.'

In [29]:
# Define the structure of the QA state
class QAState(TypedDict):
    # 'question' stores the user's input question. It can be a string or None if not provided.
    question: Optional[str]
    
    # 'context' stores relevant context about the guided project, if the question pertains to it.
    # If the question isn't related to the project, this will be None.
    context: Optional[str]
    
    # 'answer' stores the generated response or answer. It can be None until the answer is generated.
    answer: Optional[str]

In [30]:
# Create an example object
qa_state_example = QAState(
    question="What is the purpose of this guided project?",
    context="This project focuses on building a chatbot using Python.",
    answer=None
)

# Print the attributes
for key, value in qa_state_example.items():
    print(f"{key}: {value}")

question: What is the purpose of this guided project?
context: This project focuses on building a chatbot using Python.
answer: None

In [31]:
def input_validation_node(state):
    # Extract the question from the state, and strip any leading or trailing spaces
    question = state.get("question", "").strip()
    
    # If the question is empty, return an error message indicating invalid input
    if not question:
        return {"valid": False, "error": "Question cannot be empty."}
    
    # If the question is valid, return valid status
    return {"valid": True}

In [32]:
input_validation_node(qa_state_example)

{'valid': True}

In [33]:
def context_provider_node(state):
    question = state.get("question", "").lower()
    # Check if the question is related to the guided project
    if "langgraph" in question or "guided project" in question:
        context = (
            "This guided project is about using LangGraph, a Python library to design state-based workflows. "
            "LangGraph simplifies building complex applications by connecting modular nodes with conditional edges."
        )
        return {"context": context}
    # If unrelated, set context to null
    return {"context": None}

In [34]:
from langchain_ibm import ChatWatsonx

llm = ChatWatsonx(
    model_id="meta-llama/llama-4-maverick-17b-128e-instruct-fp8",
    url="https://us-south.ml.cloud.ibm.com",
    project_id="skills-network",
)

In [35]:
def llm_qa_node(state):
    # Extract the question and context from the state
    question = state.get("question", "")
    context = state.get("context", None)

    # Check for missing context and return a fallback response
    if not context:
        return {"answer": "I don't have enough context to answer your question."}

    # Construct the prompt dynamically
    prompt = f"Context: {context}\nQuestion: {question}\nAnswer the question based on the provided context."

    # Use LangChain's ChatOpenAI to get the response
    try:
        response = llm.invoke(prompt)
        return {"answer": response.content.strip()}
    except Exception as e:
        return {"answer": f"An error occurred: {str(e)}"}

In [36]:
qa_workflow = StateGraph(QAState)

In [37]:
qa_workflow.add_node("InputNode", input_validation_node)

In [38]:
qa_workflow.add_node("ContextNode", context_provider_node)

In [39]:
qa_workflow.add_node("QANode", llm_qa_node)

In [40]:
qa_workflow.set_entry_point("InputNode")

In [41]:
qa_workflow.add_edge("InputNode", "ContextNode")

In [42]:
qa_workflow.add_edge("ContextNode", "QANode")

In [43]:
qa_workflow.add_edge("QANode", END)

In [44]:
qa_app = qa_workflow.compile()

In [45]:
qa_app.invoke({"question": "What is the weather today?"})

{'question': 'What is the weather today?',
 'context': None,
 'answer': "I don't have enough context to answer your question."}

In [46]:
qa_app.invoke({"question": "What is LangGraph?"})

/opt/conda/lib/python3.12/site-packages/ibm_watsonx_ai/wml_resource.py:100: WatsonxAPIWarning: This model is a Non-IBM Product governed by a third-party license that may impose use restrictions and other obligations. By using this model you agree to its terms as identified in the following URL.
ID: disclaimer_warning
More info: https://dataplatform.cloud.ibm.com/docs/content/wsj/analyze-data/fm-models.html?context=wx
  warn(cls._build_warning_message(warning), WatsonxAPIWarning)
/opt/conda/lib/python3.12/site-packages/ibm_watsonx_ai/wml_resource.py:100: WatsonxAPIWarning: The value of 'max_tokens' for this model was set to value 1024
ID: unspecified_max_token
  warn(cls._build_warning_message(warning), WatsonxAPIWarning)

{'question': 'What is LangGraph?',
 'context': 'This guided project is about using LangGraph, a Python library to design state-based workflows. LangGraph simplifies building complex applications by connecting modular nodes with conditional edges.',
 'answer': 'LangGraph is a Python library used to design state-based workflows. It simplifies the process of building complex applications by allowing users to connect modular nodes with conditional edges.'}

In [47]:
qa_app.invoke({"question": "What is the best guided project?"})

/opt/conda/lib/python3.12/site-packages/ibm_watsonx_ai/wml_resource.py:100: WatsonxAPIWarning: This model is a Non-IBM Product governed by a third-party license that may impose use restrictions and other obligations. By using this model you agree to its terms as identified in the following URL.
ID: disclaimer_warning
More info: https://dataplatform.cloud.ibm.com/docs/content/wsj/analyze-data/fm-models.html?context=wx
  warn(cls._build_warning_message(warning), WatsonxAPIWarning)
/opt/conda/lib/python3.12/site-packages/ibm_watsonx_ai/wml_resource.py:100: WatsonxAPIWarning: The value of 'max_tokens' for this model was set to value 1024
ID: unspecified_max_token
  warn(cls._build_warning_message(warning), WatsonxAPIWarning)

{'question': 'What is the best guided project?',
 'context': 'This guided project is about using LangGraph, a Python library to design state-based workflows. LangGraph simplifies building complex applications by connecting modular nodes with conditional edges.',
 'answer': 'Based on the provided context, the best guided project is the one about using LangGraph, a Python library to design state-based workflows. This is because LangGraph simplifies building complex applications by connecting modular nodes with conditional edges, making it a valuable skill to learn for developing sophisticated applications.'}